In [ ]:
import json
from urllib.request import urlopen

import numpy as np
import pandas as pd
import seaborn as sns
import scanpy as sc
import matplotlib
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
%config InlineBackend.figure_format = 'retina'

In [ ]:
class AttrDict(dict):
    def __getattr__(self, key: str):
        try:
            return self[key]
        except KeyError:
            raise AttributeError(key) from None

with urlopen('https://phil.red/static/cellplot.json') as req:
    json_data = json.load(req, object_hook=AttrDict)

In [ ]:
json_data.data[0].keys(), json_data.data[0].transforms[0].keys(), json_data.layout.keys()

In [ ]:
data = pd.DataFrame({
    json_data.layout.xaxis.title: pd.to_datetime(json_data.data[0].x),
    json_data.layout.yaxis.title: pd.to_numeric(json_data.data[0].y, downcast='integer'),
    'Protocol': pd.Categorical(json_data.data[0].transforms[0].groups) \
        #.remove_categories(['<unknown>']),
})
data['Protocol'].cat.reorder_categories(
    sorted(data['Protocol'].cat.categories, key=lambda x: 'zzzzz' if x == '<unknown>' else x),
    inplace=True, ordered=True,
)
data.head()

In [ ]:
pal = sc.pl.palettes.default_20.copy()
pal.append(pal[len(data['Protocol'].cat.categories)-1])
pal[len(data['Protocol'].cat.categories)-1] = '#888'
pal = sns.palettes.color_palette(pal)
sns.palplot(pal)

In [ ]:
sns.set('paper', 'whitegrid', palette=pal)
matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['font.serif'] = ['Palatino', 'Palatino Linotype', 'TeX Gyre Pagella']
matplotlib.rcParams['lines.linewidth'] = .5

ax = sns.scatterplot(
    json_data.layout.xaxis.title,
    json_data.layout.yaxis.title,
    'Protocol',
    data=data,
    hue_order=data['Protocol'].cat.categories,
)

ax.set_xlim(pd.to_datetime('2009-01-01'), pd.to_datetime('2018-12-10'))
ax.set_xlabel('Submission year')

ax.set_yscale('log')
ax.set_ylim(2.4, 1e7)
ax.set_ylabel('Number of sequenced cells')

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_minor_locator(mdates.MonthLocator(7))
ax.xaxis.set_major_formatter(mticker.NullFormatter())
ax.xaxis.set_minor_formatter(mdates.DateFormatter('%Y'))
for tick in ax.xaxis.get_minor_ticks():
    tick.tick1line.set_markersize(0)
    tick.tick2line.set_markersize(0)
    tick.label1.set_horizontalalignment('center')

ax.figure.savefig('dataset-sizes.pdf')